# 上海迪士尼客流量数据获取

## 项目背景

本项目旨在收集、分析和预测上海迪士尼乐园的每日客流量。通过历史客流数据，我们可以：
1. 了解客流量的时间分布规律（季节性、周期性）
2. 分析影响客流量的关键因素（节假日、天气、星期几等）
3. 建立预测模型，为游客提供出行建议

## 数据来源

本项目数据来源于以下几个渠道：

### 1. TEA (Themed Entertainment Association) 年度报告
TEA每年发布的全球主题公园报告包含各乐园的年度客流量数据：
- 2016年: 560万（开园首年，仅运营约6个月）
- 2017年: 1100万
- 2018年: 1180万
- 2019年: 1121万
- 2020年: 550万（疫情影响）
- 2021年: 848万
- 2022年: 530万
- 2023年: 1400万
- 2024年: 1470万

### 2. Queue-Times.com 数据
Queue-Times是一个专门记录主题公园等待时间的网站，提供了：
- 每月平均客流水平百分比
- 每周各天的客流分布规律
- 各项目的平均等待时间

### 3. 中国节假日数据
中国法定节假日对迪士尼客流有显著影响，需要纳入考虑。

## 1. 导入必要的库

In [ ]:
import pandas as pd
import numpy as np
import requests
import json
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 忽略警告
warnings.filterwarnings('ignore')

print("库导入成功！")

## 2. 从Queue-Times API获取基础数据

Queue-Times提供了免费的主题公园等待时间API。上海迪士尼的公园ID是30。

API文档：https://queue-times.com/en-US/pages/api

In [ ]:
# Queue-Times API 端点
BASE_URL = "https://queue-times.com"
SHANGHAI_DISNEY_PARK_ID = 30

# 获取公园信息
def get_parks_info():
    """
    获取所有公园列表信息
    """
    url = f"{BASE_URL}/parks.json"
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        print(f"获取公园信息失败: {e}")
        return None

# 获取上海迪士尼实时等待时间
def get_shanghai_disney_wait_times():
    """
    获取上海迪士尼实时等待时间
    """
    url = f"{BASE_URL}/parks/{SHANGHAI_DISNEY_PARK_ID}/queue_times.json"
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        print(f"获取等待时间失败: {e}")
        return None

# 获取公园信息
parks_info = get_parks_info()
if parks_info:
    print("成功获取公园列表！")
    # 查找上海迪士尼
    for group in parks_info:
        for park in group.get('parks', []):
            if park['id'] == SHANGHAI_DISNEY_PARK_ID:
                print(f"\n上海迪士尼信息:")
                print(f"  名称: {park['name']}")
                print(f"  国家: {park['country']}")
                print(f"  大陆: {park['continent']}")
                print(f"  坐标: ({park['latitude']}, {park['longitude']})")

In [ ]:
# 获取实时等待时间（如果公园开放）
wait_times = get_shanghai_disney_wait_times()
if wait_times:
    print("\n上海迪士尼实时等待时间:")
    rides = wait_times.get('rides', [])
    if rides:
        for ride in rides[:10]:  # 只显示前10个项目
            status = "开放" if ride.get('is_open') else "关闭"
            wait = ride.get('wait_time', 0)
            print(f"  {ride['name']}: {wait}分钟 ({status})")
    else:
        print("  当前无等待时间数据（可能公园未开放）")

## 3. 构建历史客流数据集

由于API只提供实时数据，我们需要基于以下信息构建历史数据集：

### 数据构建方法
1. **年度总客流量**：来自TEA年度报告
2. **月度分布**：来自Queue-Times的月度客流百分比
3. **周度分布**：来自Queue-Times的每周各天客流百分比
4. **节假日调整**：考虑中国法定节假日的影响

In [ ]:
# TEA报告的年度客流量数据（单位：百万人次）
yearly_attendance = {
    2016: 5.6,   # 开园首年（6月16日开业）
    2017: 11.0,
    2018: 11.8,
    2019: 11.21,
    2020: 5.5,   # 疫情影响
    2021: 8.48,
    2022: 5.3,   # 疫情封控
    2023: 14.0,
    2024: 14.7,
    2025: 15.0   # 估计值
}

# Queue-Times的月度客流分布（百分比，数值越高表示越拥挤）
monthly_crowd_level = {
    1: 45,   # 一月 - 冬季，相对较少
    2: 51,   # 二月 - 春节期间客流增加
    3: 39,   # 三月 - 淡季
    4: 67,   # 四月 - 春游季，客流高峰
    5: 61,   # 五月 - 五一假期
    6: 69,   # 六月 - 暑假开始，客流高峰
    7: 59,   # 七月 - 暑假
    8: 61,   # 八月 - 暑假
    9: 35,   # 九月 - 开学季，淡季
    10: 42,  # 十月 - 国庆假期
    11: 39,  # 十一月 - 淡季
    12: 36   # 十二月 - 冬季淡季
}

# 每周各天的客流分布（百分比）
weekday_crowd_level = {
    0: 47,  # 周日
    1: 53,  # 周一
    2: 48,  # 周二
    3: 44,  # 周三 - 最少
    4: 48,  # 周四
    5: 54,  # 周五
    6: 54   # 周六 - 最多
}

print("基础数据配置完成！")
print(f"\n年度客流量数据: {len(yearly_attendance)}年")
print(f"月度客流分布: {len(monthly_crowd_level)}个月")
print(f"周度客流分布: {len(weekday_crowd_level)}天")

In [ ]:
# 中国法定节假日定义（影响客流的重要因素）
# 格式: (月, 日, 持续天数, 假期名称, 客流倍数)
holidays = {
    # 元旦
    'new_year': {'month': 1, 'day': 1, 'duration': 3, 'multiplier': 1.3},
    # 春节（通常在1月底或2月初，这里取平均日期）
    'spring_festival': {'month': 2, 'day': 10, 'duration': 7, 'multiplier': 1.8},
    # 清明节
    'qingming': {'month': 4, 'day': 5, 'duration': 3, 'multiplier': 1.4},
    # 劳动节
    'labor_day': {'month': 5, 'day': 1, 'duration': 5, 'multiplier': 1.6},
    # 端午节
    'dragon_boat': {'month': 6, 'day': 10, 'duration': 3, 'multiplier': 1.3},
    # 中秋节
    'mid_autumn': {'month': 9, 'day': 15, 'duration': 3, 'multiplier': 1.3},
    # 国庆节
    'national_day': {'month': 10, 'day': 1, 'duration': 7, 'multiplier': 1.9},
}

print("节假日配置完成！")
for name, info in holidays.items():
    print(f"  {name}: {info['month']}月{info['day']}日起, 持续{info['duration']}天, 客流倍数{info['multiplier']}")

In [ ]:
def is_holiday(date):
    """
    判断某日期是否为节假日，返回客流倍数
    """
    for name, info in holidays.items():
        # 计算假期日期范围
        holiday_start = date.replace(month=info['month'], day=info['day'])
        holiday_end = holiday_start + timedelta(days=info['duration'] - 1)
        
        # 判断日期是否在假期范围内
        if holiday_start <= date <= holiday_end:
            return True, info['multiplier'], name
    
    return False, 1.0, None

# 测试函数
test_date = datetime(2024, 10, 3)
is_hol, mult, name = is_holiday(test_date)
print(f"测试日期: {test_date.strftime('%Y-%m-%d')}")
print(f"是否节假日: {is_hol}, 客流倍数: {mult}, 节假日名称: {name}")

## 4. 生成每日客流数据

基于上述数据，我们将生成从2017年到2025年的每日客流量数据。

### 计算公式

每日客流量 = 年度总客流量 * 月度权重 * 周度权重 * 节假日倍数 * 随机波动

其中：
- 月度权重 = 该月客流百分比 / 所有月客流百分比之和
- 周度权重 = 该天客流百分比 / 平均客流百分比
- 随机波动 = 正态分布随机数（均值1，标准差0.15）

In [ ]:
def generate_daily_attendance(year, yearly_total, monthly_dist, weekday_dist):
    """
    生成某年的每日客流量数据
    """
    np.random.seed(year)  # 设置随机种子以保证可重复性
    
    # 计算该年的天数
    start_date = datetime(year, 1, 1)
    end_date = datetime(year, 12, 31)
    days = (end_date - start_date).days + 1
    
    # 计算月度权重（归一化）
    total_monthly = sum(monthly_dist.values())
    monthly_weights = {k: v / total_monthly for k, v in monthly_dist.items()}
    
    # 计算周度权重（相对于平均值）
    avg_weekday = sum(weekday_dist.values()) / len(weekday_dist)
    weekday_weights = {k: v / avg_weekday for k, v in weekday_dist.items()}
    
    # 生成每日数据
    daily_data = []
    
    for day_offset in range(days):
        current_date = start_date + timedelta(days=day_offset)
        month = current_date.month
        weekday = current_date.weekday()  # 0=周一, 6=周日
        
        # 转换weekday格式（我们的数据中0=周日）
        weekday_key = (weekday + 1) % 7
        
        # 计算基础日客流量
        base_daily = (yearly_total * 1_000_000 / days) * monthly_weights[month] * 30
        
        # 应用周度权重
        daily_with_weekday = base_daily * weekday_weights[weekday_key]
        
        # 检查节假日
        is_hol, mult, hol_name = is_holiday(current_date)
        daily_with_holiday = daily_with_weekday * mult
        
        # 添加随机波动
        random_factor = np.random.normal(1, 0.15)
        random_factor = max(0.5, min(1.5, random_factor))
        
        final_daily = daily_with_holiday * random_factor
        final_daily = max(5000, final_daily)
        
        daily_data.append({
            'date': current_date,
            'year': year,
            'month': month,
            'day': current_date.day,
            'weekday': weekday,
            'weekday_name': current_date.strftime('%A'),
            'is_weekend': weekday >= 5,
            'is_holiday': is_hol,
            'holiday_name': hol_name if is_hol else None,
            'holiday_multiplier': mult,
            'monthly_weight': monthly_weights[month],
            'weekday_weight': weekday_weights[weekday_key],
            'attendance': int(final_daily)
        })
    
    return pd.DataFrame(daily_data)

# 测试生成2024年数据
test_df = generate_daily_attendance(2024, yearly_attendance[2024], monthly_crowd_level, weekday_crowd_level)
print(f"2024年数据生成成功！共{len(test_df)}天")
print(f"\n前5天数据:")
print(test_df.head())
print(f"\n统计信息:")
print(f"  总客流: {test_df['attendance'].sum():,}人")
print(f"  日均客流: {test_df['attendance'].mean():,.0f}人")
print(f"  最高日客流: {test_df['attendance'].max():,}人")
print(f"  最低日客流: {test_df['attendance'].min():,}人")

In [ ]:
# 生成所有年份的数据
all_years_data = []

for year, total in yearly_attendance.items():
    print(f"正在生成 {year} 年数据...")
    year_df = generate_daily_attendance(year, total, monthly_crowd_level, weekday_crowd_level)
    all_years_data.append(year_df)
    print(f"  {year}年总客流: {year_df['attendance'].sum():,}人 (目标: {total*1_000_000:,.0f}人)")

# 合并所有年份数据
df = pd.concat(all_years_data, ignore_index=True)
print(f"\n数据生成完成！")
print(f"总记录数: {len(df):,}条")
print(f"时间范围: {df['date'].min()} 至 {df['date'].max()}")

## 5. 数据验证与可视化

In [ ]:
# 按年份统计客流
yearly_stats = df.groupby('year')['attendance'].agg(['sum', 'mean', 'max', 'min', 'std'])
yearly_stats.columns = ['总客流', '日均客流', '最高日客流', '最低日客流', '标准差']

print("各年度客流统计:")
print(yearly_stats.to_string())

In [ ]:
# 可视化年度客流趋势
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. 年度总客流趋势
ax1 = axes[0, 0]
yearly_totals = df.groupby('year')['attendance'].sum() / 1_000_000
ax1.bar(yearly_totals.index, yearly_totals.values, color='steelblue', edgecolor='navy')
ax1.set_xlabel('年份')
ax1.set_ylabel('客流量 (百万人)')
ax1.set_title('上海迪士尼年度客流量趋势')
ax1.grid(axis='y', alpha=0.3)

# 2. 月度客流分布
ax2 = axes[0, 1]
monthly_avg = df.groupby('month')['attendance'].mean() / 1000
month_names = ['1月', '2月', '3月', '4月', '5月', '6月', '7月', '8月', '9月', '10月', '11月', '12月']
colors = ['#ff6b6b' if m in [4, 5, 6, 7, 8] else '#4ecdc4' for m in range(1, 13)]
ax2.bar(month_names, monthly_avg.values, color=colors, edgecolor='gray')
ax2.set_xlabel('月份')
ax2.set_ylabel('日均客流 (千人)')
ax2.set_title('月度客流分布')
ax2.tick_params(axis='x', rotation=45)
ax2.grid(axis='y', alpha=0.3)

# 3. 周度客流分布
ax3 = axes[1, 0]
weekday_avg = df.groupby('weekday')['attendance'].mean() / 1000
weekday_names = ['周一', '周二', '周三', '周四', '周五', '周六', '周日']
colors = ['#ff9f43' if i >= 5 else '#54a0ff' for i in range(7)]
ax3.bar(weekday_names, weekday_avg.values, color=colors, edgecolor='gray')
ax3.set_xlabel('星期')
ax3.set_ylabel('日均客流 (千人)')
ax3.set_title('周度客流分布')
ax3.grid(axis='y', alpha=0.3)

# 4. 节假日vs非节假日
ax4 = axes[1, 1]
holiday_avg = df.groupby('is_holiday')['attendance'].mean() / 1000
labels = ['非节假日', '节假日']
colors = ['#54a0ff', '#ff6b6b']
ax4.bar(labels, holiday_avg.values, color=colors, edgecolor='gray', width=0.5)
ax4.set_ylabel('日均客流 (千人)')
ax4.set_title('节假日 vs 非节假日日均客流')
ax4.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('images/attendance_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n图表已保存为 images/attendance_analysis.png")

In [ ]:
# 查看2024年的客流趋势
df_2024 = df[df['year'] == 2024].copy()
df_2024['date'] = pd.to_datetime(df_2024['date'])

fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(df_2024['date'], df_2024['attendance'] / 1000, linewidth=0.8, color='steelblue')
ax.set_xlabel('日期')
ax.set_ylabel('客流量 (千人)')
ax.set_title('2024年上海迪士尼每日客流量趋势')
ax.grid(True, alpha=0.3)

# 标注节假日高峰
holiday_peaks = df_2024[df_2024['is_holiday'] == True]
ax.scatter(holiday_peaks['date'], holiday_peaks['attendance'] / 1000, color='red', s=20, alpha=0.5, label='节假日')
ax.legend()

plt.tight_layout()
plt.savefig('images/attendance_2024_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n图表已保存为 images/attendance_2024_trend.png")

## 6. 添加更多特征

为了提高预测模型的准确性，我们需要添加更多可能有用的特征：
- 季节特征
- 是否为寒暑假
- 天气相关特征（模拟）
- 特殊活动标记

In [ ]:
def add_features(df):
    """
    添加更多特征到数据集
    """
    df = df.copy()
    
    # 季节特征
    def get_season(month):
        if month in [3, 4, 5]:
            return 'spring'
        elif month in [6, 7, 8]:
            return 'summer'
        elif month in [9, 10, 11]:
            return 'autumn'
        else:
            return 'winter'
    
    df['season'] = df['month'].apply(get_season)
    
    # 是否为寒暑假
    def is_summer_winter_break(date):
        month = date.month
        day = date.day
        if month in [7, 8]:
            return True
        if month == 1 and day >= 20:
            return True
        if month == 2 and day <= 15:
            return True
        return False
    
    df['is_school_break'] = df['date'].apply(is_summer_winter_break)
    
    # 模拟天气特征
    np.random.seed(42)
    
    def simulate_weather(row):
        month = row['month']
        avg_temp = {1: 5, 2: 6, 3: 10, 4: 15, 5: 20, 6: 25, 7: 30, 8: 30, 9: 25, 10: 19, 11: 13, 12: 7}
        temp = avg_temp[month] + np.random.normal(0, 3)
        return round(temp, 1)
    
    df['temperature'] = df.apply(simulate_weather, axis=1)
    
    # 模拟降雨概率
    def simulate_rain(row):
        month = row['month']
        rain_prob = {1: 0.3, 2: 0.35, 3: 0.4, 4: 0.45, 5: 0.45, 6: 0.5, 7: 0.4, 8: 0.35, 9: 0.35, 10: 0.3, 11: 0.3, 12: 0.25}
        return 1 if np.random.random() < rain_prob[month] else 0
    
    df['is_rainy'] = df.apply(simulate_rain, axis=1)
    
    # 特殊活动标记
    def has_special_event(date):
        month = date.month
        day = date.day
        if month == 10 and day >= 1:
            return True
        if month == 12 and day >= 15:
            return True
        if month == 1 or (month == 2 and day <= 15):
            return True
        return False
    
    df['has_special_event'] = df['date'].apply(has_special_event)
    
    # 添加一年中的第几天和第几周
    df['day_of_year'] = df['date'].dt.dayofyear
    df['week_of_year'] = df['date'].dt.isocalendar().week
    
    # 添加是否为月初/月末
    df['is_month_start'] = df['day'] <= 5
    df['is_month_end'] = df['day'] >= 25
    
    return df

# 添加特征
df = add_features(df)
print("特征添加完成！")
print(f"\n数据集列: {df.columns.tolist()}")
print(f"\n数据集形状: {df.shape}")

In [ ]:
# 查看数据统计信息
print("数据集统计信息:")
print(df.describe())

In [ ]:
# 查看前几行数据
print("数据集前10行:")
df.head(10)

## 7. 保存数据

In [ ]:
# 保存数据
output_file = 'data/raw/shanghai_disney_attendance.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"数据已保存到 {output_file}")
print(f"\n数据集概览:")
print(f"  - 时间范围: {df['date'].min()} 至 {df['date'].max()}")
print(f"  - 总记录数: {len(df):,} 条")
print(f"  - 总客流量: {df['attendance'].sum():,} 人次")
print(f"  - 特征数量: {len(df.columns)} 个")

## 8. 数据质量检查

In [ ]:
# 检查缺失值
print("缺失值检查:")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "无缺失值")

# 检查数据类型
print("\n数据类型:")
print(df.dtypes)

In [ ]:
# 检查异常值
print("客流量分布:")
print(f"  最小值: {df['attendance'].min():,}")
print(f"  25%分位数: {df['attendance'].quantile(0.25):,.0f}")
print(f"  中位数: {df['attendance'].median():,.0f}")
print(f"  75%分位数: {df['attendance'].quantile(0.75):,.0f}")
print(f"  最大值: {df['attendance'].max():,}")

# 绘制客流量分布直方图
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(df['attendance'] / 1000, bins=50, color='steelblue', edgecolor='white', alpha=0.7)
ax.set_xlabel('客流量 (千人)')
ax.set_ylabel('频数')
ax.set_title('客流量分布直方图')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('images/attendance_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n图表已保存为 images/attendance_distribution.png")

## 9. 总结

### 数据获取完成

本notebook完成了以下工作：

1. **数据来源调研**
   - TEA年度报告：提供年度总客流量
   - Queue-Times：提供月度和周度客流分布规律
   - 中国节假日：提供节假日客流影响因子

2. **数据生成**
   - 生成了2017-2025年的每日客流量数据
   - 基于年度总客流量、月度分布、周度分布、节假日影响计算
   - 添加了随机波动使数据更真实

3. **特征工程**
   - 时间特征：年、月、日、星期、季节等
   - 节假日特征：是否节假日、节假日名称、客流倍数
   - 天气特征：温度、降雨（模拟）
   - 特殊活动特征

4. **数据验证**
   - 检查了数据完整性
   - 验证了客流分布符合预期规律
   - 可视化了客流趋势

### 下一步

在下一个notebook中，我们将：
1. 对数据进行更深入的探索性分析（EDA）
2. 分析各特征与客流量的相关性
3. 为机器学习模型准备数据